## Final Code 2

In [9]:
from pathlib import Path

def write_dynamic_venn_html(out_file="venn_dynamic.html"):
    html = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8" />
<title>Dynamic N‑Set Venn</title>
<meta name="viewport" content="width=device-width, initial-scale=1" />
<script src="https://cdn.anychart.com/releases/v8/js/anychart-bundle.min.js"></script>
<style>
body { font-family: Arial, sans-serif; margin: 20px; }
#container { width: 100%; height: 560px; margin-bottom: 12px; }
.panel {
  display: grid;
  grid-template-columns: 50% 50%;
  gap: 16px;
}
.card { border: 1px solid #ddd; border-radius: 6px; padding: 12px; }
.card h3 { margin: 0 0 8px 0; font-size: 16px; }
details summary { cursor: pointer; font-weight: bold; margin: 6px 0; }
ul { margin: 6px 0 0 18px; }
.mono { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Courier New", monospace; }
table { border-collapse: collapse; width: 100%; }
th, td { border: 1px solid #e0e0e0; padding: 6px 8px; text-align: left; }
th { background: #f8f8f8; }
#clickedRegionDisplay {
  border: 2px solid #555;
  background: #f0f0ff;
  padding: 15px;
  border-radius: 8px;
  margin-bottom: 16px;
}
#clickedRegionDisplay h4 { margin-top: 0; margin-bottom: 10px; font-size: 18px; color: #1a1a54; }
textarea { width: 100%; height: 120px; font-family: monospace; }
input[type="text"] { width: 100%; }
button { padding: 6px 12px; margin-top: 8px; cursor: pointer; }
.sets-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
  gap: 10px;
}
.set-block { border: 1px dashed #ccc; padding: 10px; border-radius: 6px; position: relative; }
.set-block .remove-btn {
  position: absolute; top: 8px; right: 8px;
  background: #ffe9e9; border: 1px solid #e5b3b3; border-radius: 4px; padding: 2px 8px;
}
.controls { display: flex; gap: 8px; align-items: center; margin-bottom: 8px; }
</style>
</head>
<body>
<h2>Dynamic N‑Set Venn Diagram</h2>

<div class="card" style="margin-bottom:16px;">
  <h3>Input sets</h3>
  <div class="controls">
    <button id="addSetBtn">+ Add set</button>
    <button id="buildBtn">Build Venn</button>
  </div>
  <div id="setsContainer" class="sets-grid"></div>
</div>

<div id="container"></div>

<div id="clickedRegionDisplay">
  <h4>Clicked Region Details</h4>
  <div id="regionInfo">Click a region in the Venn diagram to view its exclusive and inclusive items here.</div>
</div>

<div class="panel">
  <div class="card" style="grid-column: 1 / 2;">
    <h3>Region contents (exclusive browser)</h3>
    <div id="allRegionsExclusive"></div>
  </div>
  <div class="card" style="grid-column: 2 / 3; grid-row: 1 / span 2; overflow-y: auto;">
    <h3>Element membership</h3>
    <div id="membershipTable"></div>
  </div>
  <div class="card" style="grid-column: 1 / 2;">
    <h3>Region contents (inclusive browser)</h3>
    <div id="allRegionsInclusive"></div>
  </div>
</div>

<script>
let chart = null;
let itemsByComboExclusive = {};
let itemsByComboInclusive = {};
let membershipByItem = {};
let labelOrder = [];

function parseSet(text) {
  return new Set(
    text
      .split(/\r?\n/)      // handles \n and \r\n
      .map(s => s.trim())
      .filter(s => s.length > 0)
  );
}

function comboKey(members, orderMap) {
  return members.slice().sort((a,b) => orderMap[a] - orderMap[b]).join("&");
}

function kCombos(arr, k) {
  const res = [];
  function helper(start, combo) {
    if (combo.length === k) { res.push(combo.slice()); return; }
    for (let i=start; i<arr.length; i++) {
      combo.push(arr[i]);
      helper(i+1, combo);
      combo.pop();
    }
  }
  helper(0, []);
  return res;
}

function interSet(...sets) {
  if (sets.length === 0) return new Set();
  let res = new Set(sets[0]);
  for (let s of sets.slice(1)) {
    res = new Set([...res].filter(x => s.has(x)));
  }
  return res;
}

function addSet(label="", items="") {
  const idx = document.querySelectorAll(".set-block").length;
  const defaultLabel = label || String.fromCharCode(65 + idx) || "Set " + (idx+1);
  const el = document.createElement("div");
  el.className = "set-block";
  el.innerHTML = `
    <button class="remove-btn" title="Remove this set">&times;</button>
    <label>Label</label>
    <input type="text" class="set-label" value="${escapeHtml(defaultLabel)}">
    <label>Items (one per line)</label>
    <textarea class="set-items">${escapeHtml(items)}</textarea>
  `;
  el.querySelector(".remove-btn").addEventListener("click", () => {
    el.remove();
  });
  document.getElementById("setsContainer").appendChild(el);
}

function collectSets() {
  const blocks = Array.from(document.querySelectorAll(".set-block"));
  const labels = [];
  const sets = [];
  blocks.forEach(b => {
    const label = (b.querySelector(".set-label").value || "").trim() || `Set ${labels.length+1}`;
    labels.push(label);
    const itemsText = b.querySelector(".set-items").value || "";
    sets.push(parseSet(itemsText));
  });
  return { labels, sets };
}

function buildVenn() {
  const { labels, sets } = collectSets();
  if (labels.length === 0) {
    alert("Please add at least one set.");
    return;
  }

  labelOrder = labels.slice();
  const orderMap = {};
  labelOrder.forEach((l,i) => orderMap[l] = i);

  const setMap = {};
  labels.forEach((l,i) => { setMap[l] = sets[i]; });

  // Inclusive mapping and chart data
  itemsByComboInclusive = {};
  const allCombosKeys = [];
  for (let k=1; k<=labels.length; k++) {
    const combos = kCombos(labels, k);
    for (const combo of combos) {
      const key = comboKey(combo, orderMap);
      const inter = interSet(...combo.map(l => setMap[l]));
      const arr = Array.from(inter).sort();
      itemsByComboInclusive[key] = arr;
      allCombosKeys.push(key);
    }
  }

  // Exclusive mapping + membership
  itemsByComboExclusive = {};
  membershipByItem = {};
  const universe = new Set(sets.flatMap(s => Array.from(s)));
  Array.from(universe).sort().forEach(item => {
    const memb = [];
    labels.forEach((l,i) => { if (sets[i].has(item)) memb.push(l); });
    const key = comboKey(memb, orderMap);
    membershipByItem[item] = memb;
    if (!itemsByComboExclusive[key]) itemsByComboExclusive[key] = [];
    itemsByComboExclusive[key].push(item);
  });

  // Build chart data from inclusive sizes
  const chartData = [];
  labels.forEach(l => chartData.push({ x: l, value: (setMap[l]||new Set()).size }));
  allCombosKeys
    .filter(k => k.includes("&")) // only intersections of size >= 2
    .forEach(k => chartData.push({ x: k, value: itemsByComboInclusive[k].length }));

  renderChart(chartData);
  renderAllRegions('exclusive');
  renderAllRegions('inclusive');
  renderMembershipTable();
  document.getElementById("regionInfo").innerHTML =
    "Click a region in the Venn diagram to view its exclusive and inclusive items here.";
}

function renderChart(chartData) {
  if (!anychart) return;
  if (!chart) {
    chart = anychart.venn(chartData);
    chart.container("container");
    chart.title(null);
    chart.legend(true);
    chart.background().fill("#ffffff 0");
    chart.stroke("#444", 1.2);
    chart.labels().enabled(true);
    chart.labels().fontColor("#000").fontSize(13);
    chart.labels().format(function() {
      if (this.x.includes('&')) return '';
      return this.x;
    });
    chart.tooltip().useHtml(true).format(function() {
      var key = this.x;
      var excl = itemsByComboExclusive[key] || [];
      return "<b>" + key + "</b><br/>Count (inclusive): " + this.value +
             "<br/>Exclusive items: " + excl.length +
             (excl.length ? "<br/><span class='mono'>" +
              excl.slice(0,8).join(", ") +
              (excl.length>8 ? ", …" : "") + "</span>" : "");
    });
    chart.listen("pointClick", function(e) {
      var key = e.point.get("x");
      showRegionList(key);
    });
    chart.draw();
  } else {
    chart.data(chartData);
  }
}

function showRegionList(key) {
  var excl_arr = itemsByComboExclusive[key] || [];
  var incl_arr = itemsByComboInclusive[key] || [];
  var excl_html = "<details open><summary><b>" + key + "</b> — " +
                  excl_arr.length + " exclusive item(s)</summary>";
  if (excl_arr.length) {
    excl_html += "<ul>" + excl_arr.map(x => "<li class='mono'>" +
                 escapeHtml(x) + "</li>").join("") + "</ul>";
  } else {
    excl_html += "<em>No items exclusive to this combination.</em>";
  }
  excl_html += "</details>";
  var incl_html = "<details><summary><b>" + key + "</b> — " +
                  incl_arr.length + " inclusive item(s)</summary>";
  if (incl_arr.length) {
    incl_html += "<ul>" + incl_arr.map(x => "<li class='mono'>" +
                 escapeHtml(x) + "</li>").join("") + "</ul>";
  } else {
    incl_html += "<em>No items in this intersection (inclusive).</em>";
  }
  incl_html += "</details>";
  document.getElementById("clickedRegionDisplay").innerHTML =
    "<h4>Clicked Region: " + key + "</h4>" + excl_html + incl_html;
}

function renderAllRegions(type) {
  var boxId = type === 'exclusive' ? 'allRegionsExclusive' : 'allRegionsInclusive';
  var data = type === 'exclusive' ? itemsByComboExclusive : itemsByComboInclusive;
  var box = document.getElementById(boxId);
  var combos = Object.keys(data).sort(function(a,b){
    var da = a.split("&").length, db = b.split("&").length;
    if (da !== db) return da - db;
    return a.localeCompare(b);
  });
  var html = "";
  combos.forEach(function(key){
    var arr = data[key] || [];
    var label = type === 'exclusive' ? "exclusive item(s)" : "inclusive item(s)";
    html += "<details><summary>" + key + " — " + arr.length + " " + label + "</summary>";
    if (arr.length) {
      html += "<ul>" + arr.map(function(x){ return "<li class='mono'>" +
             escapeHtml(x) + "</li>"; }).join("") + "</ul>";
    }
    html += "</details>";
  });
  box.innerHTML = html || "<em>No items found.</em>";
}

function renderMembershipTable() {
  var box = document.getElementById("membershipTable");
  var items = Object.keys(membershipByItem).sort();
  var cols = labelOrder;
  var html = "<table><thead><tr><th>Element</th>" +
             cols.map(c => "<th>"+escapeHtml(c)+"</th>").join("") +
             "</tr></thead><tbody>";
  items.forEach(function(item){
    var memb = new Set(membershipByItem[item]);
    html += "<tr><td class='mono'>" + escapeHtml(item) + "</td>";
    cols.forEach(function(c){
      html += "<td>" + (memb.has(c) ? "✓" : "") + "</td>";
    });
    html += "</tr>";
  });
  html += "</tbody></table>";
  box.innerHTML = html;
}

function escapeHtml(s) {
  return String(s).replace(/[&<>"']/g, function(c){
    return ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]);
  });
}

// init
document.getElementById("addSetBtn").addEventListener("click", () => addSet());
document.getElementById("buildBtn").addEventListener("click", buildVenn);
// start with 4 sets A–D
["A","B","C","D"].forEach(lbl => addSet(lbl, ""));
</script>
</body>
</html>
"""
    Path(out_file).write_text(html, encoding="utf-8")
    print(f"HTML written to: {Path(out_file).resolve()}")

if __name__ == "__main__":
    write_dynamic_venn_html()

HTML written to: /home/prathamrao/Visualtization/venn_dynamic.html
